# Fase 3: Optimización de Hiperparámetros con Optuna

Este notebook visualiza gráficamente los resultados de la búsqueda de hiperparámetros ejecutada en el script `src/hyperparameter_tuning.py`.


In [1]:
# Importamos las librerías necesarias para leer y visualizar el estudio de Optuna
import os
import optuna
from optuna.visualization import plot_optimization_history, plot_param_importances, plot_parallel_coordinate

# 1. Encontrar la raíz del proyecto de forma dinámica
# os.path.abspath('') nos da el directorio actual del notebook ('notebooks/')
# os.path.dirname() sube un nivel jerárquico hasta la raíz del proyecto
NOTEBOOK_DIR = os.path.abspath('')
BASE_DIR = os.path.dirname(NOTEBOOK_DIR)

# 2. Definir la ruta exacta de la base de datos según tu arquitectura (models/trained_models/)
db_path = os.path.join(BASE_DIR, "models", "trained_models", "optuna_study.db")
study_name = 'google_ads_optimization'

print(f"Buscando base de datos en: {db_path}")

# 3. Control de flujo preventivo para evitar que SQLite cree un archivo vacío
if not os.path.exists(db_path):
    raise FileNotFoundError(
        f"¡Error! No se encontró el archivo '{db_path}'. "
        "Asegúrate de ejecutar primero el script 'src/hyperparameter_tuning.py' "
        "para inicializar y completar la base de datos de Optuna."
    )

# 4. Cargamos el estudio de forma segura desde el almacenamiento persistente
study = optuna.load_study(study_name=study_name, storage=f'sqlite:///{db_path}')

# Imprimimos el mejor resultado obtenido mediante Validación Cruzada (CV)
print("=" * 50)
print("🏆 RESULTADO GANADOR DEL ESTUDIO 🏆")
print("=" * 50)
print(f"Mejor F1-Macro Score (CV): {study.best_value:.4f}\n")

# Imprimimos la receta de configuración exacta que logró ese puntaje
print("Mejores hiperparámetros encontrados:")
for key, value in study.best_params.items():
    print(f"  -> {key}: {value}")

Buscando base de datos en: C:\Users\Carlos\Desktop\Dataset Ciencia de datos\google_ads_analytics\models\trained_models\optuna_study.db


🏆 RESULTADO GANADOR DEL ESTUDIO 🏆
Mejor F1-Macro Score (CV): 0.9694

Mejores hiperparámetros encontrados:
  -> use_pca: False
  -> classifier: LightGBM
  -> lgb_n_estimators: 200
  -> lgb_learning_rate: 0.2840428961162601


### 1. Historial de Optimización

El siguiente gráfico interactivo muestra cómo Optuna fue "aprendiendo" y mejorando su puntaje a medida que avanzaban los intentos.

* **Puntos azules**: Representan cada uno de los 30 intentos (qué F1-Score obtuvo).
* **Línea roja**: Muestra la tendencia del puntaje máximo histórico.

In [2]:
# Generamos el gráfico interactivo del historial de aprendizaje
plot_optimization_history(study)

### 2. Importancia de los Hiperparámetros

Este gráfico analiza estadísticamente qué parámetros impactaron de forma más significativa en el rendimiento final del modelo.
 

*Si la elección del 'classifier' (algoritmo) tiene una barra larga, significa que cambiar de un algoritmo a otro hizo una diferencia drástica en el rendimiento.*

In [3]:
# Generamos el gráfico de importancia relativa de cada hiperparámetro
plot_param_importances(study)

### 3. Coordenadas Paralelas (Relación entre variables)

Este gráfico es muy útil para ver visualmente las combinaciones que llevaron a los peores (colores oscuros) y a los mejores (colores amarillos/claros) resultados. 

*Tip: Puedes hacer clic y arrastrar en los ejes verticales para filtrar intentos.*

In [4]:
# Como probamos distintos algoritmos (SVM, Random Forest, etc.), no todos los intentos 
# tienen las mismas "perillas". Optuna deja el gráfico vacío porque no puede dibujar una 
# línea continua para un parámetro que no existe en todos los intentos.

# Para arreglarlo, le diremos que grafique específicamente la relación del clasificador ganador:
ganador = study.best_params['classifier']

parametros_a_graficar = ['classifier', 'use_pca']

if ganador == 'LogisticRegression':
    parametros_a_graficar.append('lr_C')
elif ganador == 'RandomForest':
    parametros_a_graficar.extend(['rf_n_estimators', 'rf_max_depth'])
elif ganador == 'SVM':
    parametros_a_graficar.extend(['svm_C', 'svm_kernel'])
elif ganador == 'XGBoost':
    parametros_a_graficar.extend(['xgb_n_estimators', 'xgb_learning_rate'])
elif ganador == 'LightGBM':
    parametros_a_graficar.extend(['lgb_n_estimators', 'lgb_learning_rate'])

# Generamos el gráfico de coordenadas paralelas solo con las variables que coinciden
plot_parallel_coordinate(study, params=parametros_a_graficar)
